# 05 — Thí nghiệm ngữ cảnh có kiểm soát: vì sao kho trộn phiên bản làm mô hình trả lời lỗi thời?

**Đề tài:** Trôi phiên bản tri thức trong trợ lý CTI dùng LLM/RAG

| Notebook | Việc | Đầu ra |
|---|---|---|
| `01`–`03` | Ground truth thay đổi, benchmark theo phiên bản, cặp câu hỏi trước/sau | `drift/`, `benchmark/` |
| `04_pilot_llm_rag` | Pilot: kho trộn làm tỉ lệ trả lời lỗi thời tăng từ 4–11% lên 83–91% | `results/pilot-18.1-19.2/` |
| **`05_ngu_canh_kiem_soat` (notebook này)** | Tách **nguyên nhân** bằng ngữ cảnh do ta tự dựng | `results/ctx-18.1-19.2/` |

## Câu hỏi

Pilot cho thấy: khi kho RAG chứa **cả** v18.1 lẫn v19.2, mô hình trả lời theo bản cũ ở 83–91% câu có đáp án đổi. Nhưng trong pilot, BM25 thường xếp đoạn cũ **trước** đoạn mới (hoà điểm). Vì vậy chưa rõ mô hình chọn bản cũ do:

- **vị trí** (đọc đoạn đầu tiên),
- **sự có mặt** của đoạn cũ,
- hay **bản thân nội dung cũ quen thuộc hơn** (ví dụ "Defense Evasion" quen hơn "Defense Impairment").

Notebook này **không dùng BM25 để chọn bằng chứng**. Với mỗi câu hỏi, ta tự đặt đúng đoạn bằng chứng cũ (O) và/hoặc mới (N) vào ngữ cảnh theo thứ tự định trước. Mọi điều kiện có **cùng số đoạn (4)**. Chỗ còn trống được lấp bằng **cùng các đoạn nhiễu** (đoạn v19.2 liên quan đến câu hỏi nhưng không chứa đáp án). Nhờ vậy, mọi chênh lệch giữa hai điều kiện chỉ đến từ yếu tố được thay đổi.

## Bảy điều kiện

| Điều kiện | Ngữ cảnh (bằng chứng trước, đoạn nhiễu sau) | Ý nghĩa |
|---|---|---|
| `new` | N | chỉ bằng chứng mới (kho sạch, cận trên) |
| `old` | O | chỉ bằng chứng cũ (kho chưa cập nhật) |
| `old_new` | O, N | có cả hai, **cũ trước** |
| `new_old` | N, O | có cả hai, **mới trước** |
| `tag_old_new` | O\*, N\* | như `old_new`, mọi đoạn **gắn nhãn phiên bản** |
| `tag_new_old` | N\*, O\* | như `new_old`, có nhãn phiên bản |
| `old_changelog` | O, L | kho cũ + **ghi chú thay đổi** (L) lấy từ release notes |

- **Nhãn phiên bản (\*):** dòng `[Source: MITRE ATT&CK Enterprise v18.1 (superseded)]` hoặc `[Source: MITRE ATT&CK Enterprise v19.2 (current release)]` ở đầu đoạn.
- **Ghi chú thay đổi (L):** một câu sinh tất định từ ground truth (notebook 01/03). Ví dụ: *"In ATT&CK v19, technique T1222 (File and Directory Permissions Modification) moved from tactic(s) Defense Evasion (TA0005) to Defense Impairment (TA0112)."* Với mã bị thu hồi: *"…T1562.004 … was revoked and replaced by T1686 …"*.

`tag_*` và `old_changelog` là hai **phương pháp đơn giản nhất** của Trụ 3: gắn siêu dữ liệu phiên bản, và bổ sung nhật ký thay đổi. Ở đây ta đo **cận trên** của chúng: bằng chứng được đặt sẵn, không phụ thuộc truy hồi.

## Giả thuyết (định trước, trước khi xem kết quả)

| | Giả thuyết | So sánh (cùng câu hỏi, ghép cặp) |
|---|---|---|
| **H1** | Vị trí: đoạn đứng trước được ưu tiên | drift(`old_new`) − drift(`new_old`) > 0 |
| **H2** | Chỉ riêng sự có mặt của đoạn cũ đã gây lỗi thời | drift(`new_old`) − drift(`new`) > 0 |
| **H3** | Gắn nhãn phiên bản giảm lỗi thời | drift(`old_new`) − drift(`tag_old_new`) > 0, và drift(`new_old`) − drift(`tag_new_old`) > 0 |
| **H4** | Ghi chú thay đổi sửa được kho cũ | drift(`old`) − drift(`old_changelog`) > 0, **trừ** câu D1 có mã bị thu hồi (ghi chú chỉ cho mã thay thế, không cho tactic, nên ngữ cảnh không đủ để trả lời đúng) |
| **H5** | Thiên lệch về nhãn quen thuộc (khám phá): tactic hoàn toàn mới (TA0112) bị bỏ qua nhiều hơn | trong `new_old`, chỉ xét kỹ thuật **còn hiệu lực**: drift(NEW_TACTIC) − drift(EXISTING_TACTICS) > 0. Đây là so sánh **giữa các mục khác nhau**, nên chỉ mang tính khám phá. |

`drift` = `drift_ext` (notebook 04): tỉ lệ trả lời theo bản cũ trên câu có đáp án đổi. Riêng câu "cùng mã, đổi tên" (TA0005) có chỉ số riêng là tỉ lệ dùng **tên cũ**.

## Dữ liệu và lưu ý phương pháp

- **Truy vấn:** câu hỏi về **bản hiện hành** của các mục pilot, mẫu D1, D2, D4. D3 bị loại vì bằng chứng gồm nhiều đoạn mitigation, không đặt gọn được vào 4 đoạn. Có 3 nhóm:
  - `changed`: đáp án đổi, 224 truy vấn;
  - `label`: cùng mã TA0005 nhưng đổi tên, 160;
  - `stable`: đối chứng, 240.
- **Đây là thí nghiệm khám phá cơ chế** trên lần chuyển phiên bản dùng để phát triển (v18.1 → v19.2). Kết luận chính thức của bài sẽ được **kiểm định lại trên các lần chuyển phiên bản khác** (ví dụ v15→v16, v16→v17), không dùng lại các mục này.
- **Hai mô hình và cách chạy như notebook 04:** cùng lời dặn hệ thống, cùng lược đồ JSON, cùng module suy luận `ctidrift_infer.py`. Khác biệt duy nhất về định dạng: notebook này dùng hàm `clip` đã sửa, giữ nguyên xuống dòng của đoạn dài. Bản pilot làm mất xuống dòng, nên con số của hai notebook không so sánh trực tiếp từng điểm phần trăm. Thời gian chạy khoảng 15–25 phút mỗi mô hình trên RTX 4070 (mô hình đã có sẵn trong `/data/hf`).
- **Cách hỏi chính là `current`** (có nói rõ phiên bản và gợi ý "nếu mã đã bị thay thế…"). Với `unversioned`, khi ngữ cảnh chứa cả hai phiên bản, một câu trả lời theo bản cũ **không hẳn sai** (câu hỏi không nói hỏi bản nào). Vì vậy kiểm định giả thuyết dùng `current`; `unversioned` được báo cáo riêng.
- **Khoảng tin cậy bootstrap theo cụm "dòng dõi"** (`lineage`): các câu D1, D2, D4 về cùng một kỹ thuật (hoặc mã cũ cùng mã thay thế) được tính là một cụm.
- **Nhãn phiên bản gắn cho mọi đoạn**, kể cả đoạn nhiễu (như một chỉ mục được gắn nhãn toàn bộ). H3 đo tác động của "gắn nhãn cả chỉ mục", không chỉ riêng đoạn bằng chứng.

### Nhật ký phiên bản

| Phiên bản | Ngày | Thay đổi |
|---|---|---|
| 1.0 | 12/10/2026 | Bản đầu tiên |
| 1.1 | 12/10/2026 | Sửa theo kiểm tra độc lập (trước khi chạy thật): đoạn nhiễu không được chứa mã/tên của đáp án hai phiên bản; kiểm định trên cách hỏi `current`, `unversioned` báo cáo riêng; bootstrap theo cụm dòng dõi; H4 loại D1 thu hồi và có bảng theo mẫu × thu hồi; H5 chỉ xét kỹ thuật còn hiệu lực. |

## 1. Cấu hình và kiểm tra đầu vào

Notebook này dùng lại `ctidrift_infer.py` và `ctidrift_scoring.py` do notebook 04 và 03 ghi ra. Nếu thiếu, hãy chạy lại các notebook đó.

In [1]:
from pathlib import Path
import os, sys, json, re, math, time, hashlib, subprocess, collections, datetime, zipfile, signal
import numpy as np
import pandas as pd

V_OLD, V_NEW = "18.1", "19.2"
ROOT    = Path.cwd()
DRIFT   = ROOT / "benchmark" / f"drift-{V_OLD}-{V_NEW}"
OUT     = ROOT / "results" / f"ctx-{V_OLD}-{V_NEW}"
PROMPTS = OUT / "prompts"
SMOKE_PROMPTS = OUT / "prompts_smoke"
RAW     = OUT / "raw"
LOGS    = OUT / "logs"
for p in (OUT, PROMPTS, SMOKE_PROMPTS, RAW, LOGS):
    p.mkdir(parents=True, exist_ok=True)

HF_HOME         = "/data/hf"
VLLM_CACHE_ROOT = "/data/vllm-cache"
MODELS = [
    dict(tag="qwen3-8b", repo="Qwen/Qwen3-8B-AWQ",
         revision="4da05a8edb55c6046cce958586c33b61da07bb79", released="2025-05",
         chat_template_kwargs={"enable_thinking": False}),
    dict(tag="llama3.1-8b", repo="hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
         revision="db1f81ad4b8c7e39777509fac66c652eb0a52f91", released="2024-07",
         chat_template_kwargs={}),
]
CONDITIONS = {                       # thứ tự các thành phần bằng chứng trong ngữ cảnh
    "new":           ["N"],
    "old":           ["O"],
    "old_new":       ["O", "N"],
    "new_old":       ["N", "O"],
    "tag_old_new":   ["O*", "N*"],
    "tag_new_old":   ["N*", "O*"],
    "old_changelog": ["O", "L"],
}
TEMPLATES       = ["D1_tech_tactic", "D2_subtech_parent", "D4_id_currency"]
N_PASSAGES      = 4          # tổng số đoạn trong mọi ngữ cảnh (bằng chứng + đoạn nhiễu)
MAX_CHUNK_WORDS = 250
MAX_MODEL_LEN   = 6144
MAX_TOKENS      = 160
GPU_MEM         = 0.85
SEED            = 2026
BOOT_B          = 2000
SMOKE_PER_GROUP = 2
CTX_VERSION     = "1.1"

REQUIRED = {
    "pilot_queries":  DRIFT / "pilot_queries.jsonl",
    "pilot_items":    DRIFT / "pilot_items.jsonl",
    "drift_manifest": DRIFT / "manifest.json",
    "tactic_alias":   DRIFT / "tactic_alias.json",
    "corpus_old":     ROOT / "benchmark" / f"attack-{V_OLD}" / "corpus.jsonl",
    "corpus_new":     ROOT / "benchmark" / f"attack-{V_NEW}" / "corpus.jsonl",
    "scoring_module": ROOT / "ctidrift_scoring.py",
    "infer_module":   ROOT / "ctidrift_infer.py",
}
missing = [f"{k}: {p}" for k, p in REQUIRED.items() if not p.exists()]
assert not missing, "Thiếu đầu vào — hãy chạy notebook 02, 03, 04 trước:\n  " + "\n  ".join(missing)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

DM = json.loads(REQUIRED["drift_manifest"].read_text())
for k in ("pilot_queries", "pilot_items"):
    assert DM["outputs"][f"{k}.jsonl"] == sha256(REQUIRED[k]), f"{k}.jsonl không khớp manifest notebook 03"
assert "structured_outputs_config" in REQUIRED["infer_module"].read_text(), \
    "ctidrift_infer.py là bản cũ — hãy chạy lại notebook 04 bản 1.1 (ô %%writefile) trước"

ENV = {**os.environ, "HF_HOME": HF_HOME, "HF_HUB_CACHE": f"{HF_HOME}/hub", "VLLM_CACHE_ROOT": VLLM_CACHE_ROOT,
       "VLLM_USE_FLASHINFER_SAMPLER": "0", "TOKENIZERS_PARALLELISM": "false"}
ENV.pop("HF_HUB_OFFLINE", None)

MANIFEST = {"ctx_version": CTX_VERSION, "v_old": V_OLD, "v_new": V_NEW,
            "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
            "config": dict(CONDITIONS=CONDITIONS, TEMPLATES=TEMPLATES, N_PASSAGES=N_PASSAGES,
                           MAX_CHUNK_WORDS=MAX_CHUNK_WORDS, MAX_MODEL_LEN=MAX_MODEL_LEN, MAX_TOKENS=MAX_TOKENS,
                           GPU_MEM=GPU_MEM, SEED=SEED, BOOT_B=BOOT_B),
            "models": MODELS,
            "inputs": {k: {"path": str(p.relative_to(ROOT)), "sha256": sha256(p)} for k, p in REQUIRED.items()}}
for k, v in MANIFEST["inputs"].items():
    print(f"{k:15s} {v['sha256'][:12]}…  {v['path']}")

pilot_queries   1382ad00e2b4…  benchmark/drift-18.1-19.2/pilot_queries.jsonl
pilot_items     d76133e1cd33…  benchmark/drift-18.1-19.2/pilot_items.jsonl
drift_manifest  6abfab5da091…  benchmark/drift-18.1-19.2/manifest.json
tactic_alias    a5efc7448153…  benchmark/drift-18.1-19.2/tactic_alias.json
corpus_old      d435b77a7e8f…  benchmark/attack-18.1/corpus.jsonl
corpus_new      0198ec96b46b…  benchmark/attack-19.2/corpus.jsonl
scoring_module  56b3cb0eb0c8…  ctidrift_scoring.py
infer_module    833c7fe70aba…  ctidrift_infer.py


## 2. Chọn truy vấn và bằng chứng

Với mỗi truy vấn ta xác định:

- **O** = đoạn bằng chứng ở kho v18.1 (`evidence[18.1]` của notebook 03): đoạn của **mã cũ** được hỏi.
- **N** = đoạn bằng chứng ở kho v19.2: đoạn của **mã hiện hành**. Với mã bị thu hồi, đây là đoạn của mã thay thế. Ở D4, đoạn này **không** nói gì tới mã cũ, nên chỉ ghi chú thay đổi mới nối được hai mã.
- **Nhóm:** `changed` (đáp án đổi), `label` (D1, cùng mã TA0005 nhưng đổi tên), `stable` (đối chứng).

Kiểm tra: mọi truy vấn phải có đủ O và N, và đoạn bằng chứng phải chứa đúng đáp án của phiên bản tương ứng.

In [2]:
import importlib, ctidrift_scoring
importlib.reload(ctidrift_scoring)
from ctidrift_scoring import score_drift

QUERIES_ALL = [json.loads(l) for l in open(REQUIRED["pilot_queries"], encoding="utf-8")]
ITEMS = {it["item_id"]: it for it in map(json.loads, open(REQUIRED["pilot_items"], encoding="utf-8"))}
ALIAS_FILE = json.loads(REQUIRED["tactic_alias"].read_text())
TACTIC_ALIAS = ALIAS_FILE["alias"]
CORP = {V_OLD: {c["_id"]: c for c in map(json.loads, open(REQUIRED["corpus_old"], encoding="utf-8"))},
        V_NEW: {c["_id"]: c for c in map(json.loads, open(REQUIRED["corpus_new"], encoding="utf-8"))}}

def group_of(q):
    if q["answer_changed"]:
        return "changed"
    if q["pair_class"] == "LABEL_CHANGED":
        return "label"
    if q["pair_class"] == "STABLE":
        return "stable"
    return None

def evidence_new(q, it):
    ev = q["evidence"][V_NEW]
    return ev if ev else [f"{s}#0" for s in it["anchor_new"] if f"{s}#0" in CORP[V_NEW]]

QUERIES, EV = [], {}
for q in QUERIES_ALL:
    g = group_of(q)
    if q["direction"] != "current" or q["template"] not in TEMPLATES or g is None:
        continue
    it = ITEMS[q["item_id"]]
    eo, en = q["evidence"][V_OLD], evidence_new(q, it)
    assert eo and en, f"{q['query_id']}: thiếu bằng chứng O hoặc N"
    assert all(e in CORP[V_OLD] for e in eo) and all(e in CORP[V_NEW] for e in en)
    QUERIES.append({**q, "group": g})
    EV[q["query_id"]] = (eo, en)
QBYID = {q["query_id"]: q for q in QUERIES}

# Kiểm tra: đoạn O chứa đáp án v18.1, đoạn N chứa đáp án v19.2 (trừ D4 thu hồi: N là đoạn của mã thay thế)
bad = []
for q in QUERIES:
    eo, en = EV[q["query_id"]]
    t_old = "\n".join(CORP[V_OLD][e]["text"] for e in eo)
    t_new = "\n".join(CORP[V_NEW][e]["text"] for e in en)
    if not all(re.search(re.escape(g) + r"(?![.\d])", t_old) for g in q["other_gold"] if q["group"] == "changed"):
        bad.append((q["query_id"], "O"))
    if not all(re.search(re.escape(g) + r"(?![.\d])", t_new) for g in q["gold"]):
        bad.append((q["query_id"], "N"))
assert not bad, f"Bằng chứng không chứa đáp án: {bad[:10]}"
print(f"{len(QUERIES)} truy vấn:", dict(collections.Counter((q['template'][:2], q['group']) for q in QUERIES)))
print("✓ Mọi đoạn O chứa đáp án v18.1 và mọi đoạn N chứa đáp án v19.2")

624 truy vấn: {('D1', 'stable'): 80, ('D1', 'label'): 160, ('D1', 'changed'): 174, ('D2', 'stable'): 80, ('D2', 'changed'): 16, ('D4', 'stable'): 80, ('D4', 'changed'): 34}
✓ Mọi đoạn O chứa đáp án v18.1 và mọi đoạn N chứa đáp án v19.2


## 3. Ghi chú thay đổi (L) và đoạn nhiễu

**Ghi chú thay đổi** được sinh **tất định** từ thông tin của notebook 03 (lớp thay đổi, đáp án hai phiên bản, mã thay thế). Đây chính là nội dung có trong release notes chính thức. Ba mẫu câu:

- mã bị thu hồi: *"In MITRE ATT&CK v19, technique {mã cũ} ({tên cũ}) was revoked and replaced by {mã mới} ({tên mới})."*
- đổi tactic: *"In MITRE ATT&CK v19, the tactic assignment of technique {mã} ({tên}) changed from {tactic cũ} to {tactic mới}."*
- đổi tên tactic: *"In MITRE ATT&CK v19, tactic TA0005 was renamed from Defense Evasion to Stealth."*

Nhóm `stable` không có thay đổi nên không có ghi chú: ở điều kiện `old_changelog`, chỗ đó được lấp bằng một đoạn nhiễu.

**Đoạn nhiễu:** các đoạn v19.2 được BM25 xếp cao nhất cho câu hỏi, **loại** mọi đoạn có nhắc tới:

- mã được hỏi, mã thay thế;
- **bất kỳ mã nào trong đáp án của hai phiên bản**;
- với câu hỏi tactic, thêm **tên** các tactic đó (ví dụ "Defense Evasion", "Stealth").

Nhờ vậy đoạn nhiễu **không lộ đáp án**, kể cả gián tiếp. Với mỗi truy vấn, danh sách đoạn nhiễu cố định và dùng chung cho mọi điều kiện.

In [3]:
_old_names = {n for n in TACTIC_ALIAS if n not in ALIAS_FILE["new_only_labels"]}
_new_names = {n for n in TACTIC_ALIAS if n not in ALIAS_FILE["old_only_labels"]}
OLD_NAME = {TACTIC_ALIAS[n]: n for n in sorted(_old_names)}
NEW_NAME = {TACTIC_ALIAS[n]: n for n in sorted(_new_names)}
RENAMED_TACTICS = {t: (OLD_NAME[t], NEW_NAME[t]) for t in OLD_NAME if t in NEW_NAME and OLD_NAME[t] != NEW_NAME[t]}
print("Tactic đổi tên:", RENAMED_TACTICS)

def changelog_note(q):
    it = ITEMS[q["item_id"]]
    major = V_NEW.split(".")[0]
    if q["group"] == "label":
        t, (a, b) = next(iter(RENAMED_TACTICS.items()))
        return f"In MITRE ATT&CK v{major}, tactic {t} was renamed from {a} to {b}."
    if q["group"] != "changed":
        return None
    if it["anchor_status_new"] != "active":
        names = it["name_new"].split(" / ")
        repl = ", ".join(f"{s} ({n})" for s, n in zip(it["anchor_new"], names))
        return (f"In MITRE ATT&CK v{major}, technique {it['anchor_old']} ({it['name_old']}) was revoked "
                f"and replaced by {repl}.")
    return (f"In MITRE ATT&CK v{major}, the tactic assignment of technique {it['anchor_old']} ({it['name_old']}) "
            f"changed from {it['gold_display_old']} to {it['gold_display_new']}.")

NOTES = {q["query_id"]: changelog_note(q) for q in QUERIES}
assert all(NOTES[q["query_id"]] for q in QUERIES if q["group"] != "stable")
print("Ví dụ ghi chú:")
for qid in ["D1-T1222|current", "D4-T1562.004|current", "D1-T1027|current"]:
    if qid in NOTES:
        print("  ", NOTES[qid])

Tactic đổi tên: {'TA0005': ('Defense Evasion', 'Stealth')}
Ví dụ ghi chú:
   In MITRE ATT&CK v19, the tactic assignment of technique T1222 (File and Directory Permissions Modification) changed from Defense Evasion (TA0005) to Defense Impairment (TA0112).
   In MITRE ATT&CK v19, technique T1562.004 (Disable or Modify System Firewall) was revoked and replaced by T1686 (Disable or Modify System Firewall).
   In MITRE ATT&CK v19, tactic TA0005 was renamed from Defense Evasion to Stealth.


In [4]:
TOKEN_RE = re.compile(r"[a-z0-9]+(?:\.[0-9]+)*")
def tokenize(text):
    return TOKEN_RE.findall(text.lower())

class BM25:                            # giống hệt notebook 04
    def __init__(self, docs, k1=1.2, b=0.75):
        self.ids = sorted(docs)
        n = len(self.ids)
        toks = [collections.Counter(tokenize(docs[i])) for i in self.ids]
        dl = np.array([sum(t.values()) for t in toks], dtype=float)
        avgdl = dl.mean()
        post = collections.defaultdict(lambda: ([], []))
        for j, tf in enumerate(toks):
            for t, c in tf.items():
                post[t][0].append(j)
                post[t][1].append(c)
        self.post = {}
        for t, (idx, tf) in post.items():
            idx, tf = np.array(idx), np.array(tf, dtype=float)
            idf = math.log(1 + (n - len(idx) + 0.5) / (len(idx) + 0.5))
            self.post[t] = (idx, idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[idx] / avgdl)))
        self.n = n

    def search(self, query, k=5):
        scores = np.zeros(self.n)
        for t in sorted(set(tokenize(query))):
            if t in self.post:
                idx, w = self.post[t]
                scores[idx] += w
        cand = np.nonzero(scores)[0]
        order = cand[np.lexsort((cand, -scores[cand]))][:k]
        return [(self.ids[j], float(scores[j])) for j in order]

INDEX_NEW = BM25({i: d["title"] + "\n" + d["text"] for i, d in CORP[V_NEW].items()})

def banned_terms(q):
    # Mọi mã có thể "lộ" đáp án: mã được hỏi, mã thay thế, đáp án hai phiên bản; với câu tactic thêm tên tactic
    it = ITEMS[q["item_id"]]
    ids = {it["anchor_old"], *it["anchor_new"], *q["gold"], *q["other_gold"]}
    names = set()
    if q["answer_type"] == "tactic_set":
        for t in set(q["gold"]) | set(q["other_gold"]):
            names |= {OLD_NAME.get(t), NEW_NAME.get(t)} - {None}
    return ids, names

def distractors(q, k):
    it = ITEMS[q["item_id"]]
    ids, names = banned_terms(q)
    id_re = re.compile(r"\b(?:" + "|".join(re.escape(b) for b in sorted(ids)) + r")(?![.\d])")
    name_re = re.compile(r"\b(?:" + "|".join(re.escape(n) for n in sorted(names)) + r")\b", re.I) if names else None
    eo, en = EV[q["query_id"]]
    out = []
    for cid, _ in INDEX_NEW.search(q["question"], 3000):
        d = CORP[V_NEW][cid]
        if (cid in en or d["metadata"]["doc_id"] in ids or id_re.search(d["text"])
                or (name_re and name_re.search(d["text"]))):
            continue
        out.append(cid)
        if len(out) == k:
            break
    assert len(out) == k, f"{q['query_id']}: không đủ đoạn nhiễu"
    return out

DISTRACT = {q["query_id"]: distractors(q, N_PASSAGES) for q in QUERIES}
# Kiểm tra độc lập: không đoạn nhiễu nào chứa mã hay tên tactic có thể lộ đáp án
leaks = []
for q in QUERIES:
    ids, names = banned_terms(q)
    for cid in DISTRACT[q["query_id"]]:
        t = CORP[V_NEW][cid]["text"]
        if any(re.search(re.escape(i) + r"(?![.\d])", t) for i in ids) or any(n.lower() in t.lower() for n in names):
            leaks.append((q["query_id"], cid))
assert not leaks, f"Đoạn nhiễu lộ đáp án: {leaks[:5]}"
print(f"✓ Đã chọn {N_PASSAGES} đoạn nhiễu cố định cho mỗi truy vấn, không đoạn nào chứa mã/tên của đáp án hai phiên bản")

✓ Đã chọn 4 đoạn nhiễu cố định cho mỗi truy vấn, không đoạn nào chứa mã/tên của đáp án hai phiên bản


## 4. Dựng prompt cho 7 điều kiện

**Lời dặn hệ thống, yêu cầu theo kiểu đáp án và lược đồ JSON giống hệt notebook 04.** Ngữ cảnh gồm đúng `N_PASSAGES = 4` đoạn: các thành phần bằng chứng theo thứ tự của điều kiện, sau đó là đoạn nhiễu cho đủ 4.

Ô cuối mục này **tự kiểm tra** từng prompt:

- đúng số đoạn;
- đúng thứ tự O/N;
- nhãn phiên bản chỉ có ở điều kiện `tag_*`;
- ghi chú thay đổi chỉ có ở `old_changelog`;
- các đoạn nhiễu giống nhau giữa các điều kiện, xét theo vị trí tương ứng.

In [5]:
SYSTEM = ("You are a cyber threat intelligence (CTI) assistant with expertise in the MITRE ATT&CK Enterprise framework. "
          "Answer the user's question accurately. Respond only with a JSON object of the form {\"answer\": [...]}. "
          "Put only the requested items in the list. Use an empty list if you do not know.")
INSTR = {
    "tactic_set":     'List each tactic as its ATT&CK tactic ID followed by its name, for example "TA0001 Initial Access".',
    "technique_set":  "List ATT&CK technique IDs only (format T1234 or T1234.001).",
    "mitigation_set": "List ATT&CK mitigation IDs only (format M1234).",
}
PATTERN = {"technique_set": "^T[0-9]{4}(\\.[0-9]{3})?$", "mitigation_set": "^M[0-9]{4}$",
           "tactic_set": "^[A-Za-z0-9 ,.()&/'-]{1,60}$"}
TAG = {V_OLD: f"[Source: MITRE ATT&CK Enterprise v{V_OLD} (superseded)]",
       V_NEW: f"[Source: MITRE ATT&CK Enterprise v{V_NEW} (current release)]"}

def schema_for(answer_type):
    return {"type": "object",
            "properties": {"answer": {"type": "array", "items": {"type": "string", "pattern": PATTERN[answer_type]},
                                      "maxItems": 8}},
            "required": ["answer"], "additionalProperties": False}

def clip(text, n):
    # Giữ nguyên định dạng gốc (xuống dòng), chỉ cắt sau từ thứ n
    words = list(re.finditer(r"\S+", text))
    return text if len(words) <= n else text[:words[n - 1].end()] + " …"

def passages(q, cond):
    # Trả về danh sách (loại, phiên bản, văn bản) theo đúng thứ tự trong ngữ cảnh
    eo, en = EV[q["query_id"]]
    tagged = cond.startswith("tag_")
    out = []
    for comp in CONDITIONS[cond]:
        kind = comp.rstrip("*")
        if kind == "O":
            out += [("O", V_OLD, CORP[V_OLD][e]["text"]) for e in eo]
        elif kind == "N":
            out += [("N", V_NEW, CORP[V_NEW][e]["text"]) for e in en]
        elif kind == "L" and NOTES[q["query_id"]]:
            out.append(("L", None, NOTES[q["query_id"]]))
    n_fill = max(0, N_PASSAGES - len(out))
    out += [("D", V_NEW, CORP[V_NEW][d]["text"]) for d in DISTRACT[q["query_id"]][:n_fill]]
    return [(k, v, (TAG[v] + "\n" if tagged and v else "") + clip(t, MAX_CHUNK_WORDS)) for k, v, t in out]

def build_messages(q, cond):
    ctx = "\n\n".join(f"[{i + 1}] {t}" for i, (_, _, t) in enumerate(passages(q, cond)))
    user = ("Use the following context passages from a CTI knowledge base if they are relevant.\n\n"
            f"{ctx}\n\nQuestion: {q['question']}\n{INSTR[q['answer_type']]}")
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

LAYOUT = {}
for cond in CONDITIONS:
    rows = []
    for q in QUERIES:
        ps = passages(q, cond)
        LAYOUT[(cond, q["query_id"])] = [k for k, _, _ in ps]
        rows.append(dict(query_id=q["query_id"], condition=cond, answer_type=q["answer_type"],
                         messages=build_messages(q, cond), schema=schema_for(q["answer_type"]),
                         layout=LAYOUT[(cond, q["query_id"])]))
    with open(PROMPTS / f"{cond}.jsonl", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False, sort_keys=True) + "\n")
MANIFEST["prompts"] = {c: sha256(PROMPTS / f"{c}.jsonl") for c in CONDITIONS}
print(pd.Series({c: collections.Counter(tuple(LAYOUT[(c, q['query_id'])]) for q in QUERIES).most_common(2)
                 for c in CONDITIONS}).to_string())

new                                   [((N, D, D, D), 624)]
old                                   [((O, D, D, D), 624)]
old_new                               [((O, N, D, D), 624)]
new_old                               [((N, O, D, D), 624)]
tag_old_new                           [((O, N, D, D), 624)]
tag_new_old                           [((N, O, D, D), 624)]
old_changelog    [((O, L, D, D), 384), ((O, D, D, D), 240)]


In [6]:
# Tự kiểm tra cấu trúc prompt
errs = []
for q in QUERIES:
    eo, en = EV[q["query_id"]]
    for cond, comps in CONDITIONS.items():
        ps = passages(q, cond)
        kinds = [k for k, _, _ in ps]
        exp = []
        for c in comps:
            k = c.rstrip("*")
            exp += ["O"] * len(eo) if k == "O" else ["N"] * len(en) if k == "N" else (["L"] if NOTES[q["query_id"]] else [])
        if kinds[:len(exp)] != exp or set(kinds[len(exp):]) - {"D"} or len(ps) != max(N_PASSAGES, len(exp)):
            errs.append((q["query_id"], cond, "bố cục"))
        text = build_messages(q, cond)[1]["content"]
        if cond.startswith("tag_") != ("[Source: MITRE ATT&CK" in text):
            errs.append((q["query_id"], cond, "nhãn phiên bản"))
        note = NOTES[q["query_id"]]
        if (cond == "old_changelog" and bool(note)) != (bool(note) and note in text):
            errs.append((q["query_id"], cond, "ghi chú"))
    # đoạn nhiễu dùng chung: điều kiện 2 bằng chứng có cùng đoạn nhiễu
    d1 = [t for k, _, t in passages(q, "old_new") if k == "D"]
    d2 = [t for k, _, t in passages(q, "new_old") if k == "D"]
    if d1 != d2:
        errs.append((q["query_id"], "*", "đoạn nhiễu khác nhau"))
assert not errs, errs[:10]
print(f"✓ {len(QUERIES) * len(CONDITIONS)} prompt đúng cấu trúc")
print("\nVí dụ (tag_new_old, D1-T1222|current):\n" + build_messages(QBYID["D1-T1222|current"], "tag_new_old")[1]["content"][:1500]
      if "D1-T1222|current" in QBYID else "")

✓ 4368 prompt đúng cấu trúc

Ví dụ (tag_new_old, D1-T1222|current):
Use the following context passages from a CTI knowledge base if they are relevant.

[1] [Source: MITRE ATT&CK Enterprise v19.2 (current release)]
ATT&CK Technique T1222: File and Directory Permissions Modification
Tactics: Defense Impairment (TA0112).
Platforms: ESXi, Linux, macOS, Windows.
Adversaries may modify file or directory permissions/attributes to evade access control lists (ACLs) and access protected files. File and directory permissions are commonly managed by ACLs configured by the file or directory owner, or users with the appropriate permissions. File and directory ACL implementations vary by platform, but generally explicitly designate which users or groups can perform which actions (read, write, execute, etc.).

Modifications may include changing specific access rights, which may require taking ownership of a file or directory and/or elevated permissions depending on the file or directory’s existing per

In [7]:
smoke_ids = []
for key in sorted({(q["template"], q["group"]) for q in QUERIES}):
    smoke_ids += [q["query_id"] for q in QUERIES if (q["template"], q["group"]) == key][:SMOKE_PER_GROUP]
smoke_set = set(smoke_ids)
for cond in CONDITIONS:
    with open(PROMPTS / f"{cond}.jsonl", encoding="utf-8") as fi, open(SMOKE_PROMPTS / f"{cond}.jsonl", "w", encoding="utf-8") as fo:
        for line in fi:
            if json.loads(line)["query_id"] in smoke_set:
                fo.write(line)
print(f"Bộ chạy thử: {len(smoke_ids)} truy vấn/điều kiện")

Bộ chạy thử: 14 truy vấn/điều kiện


## 5. Chạy suy luận

Dùng lại `ctidrift_infer.py` (notebook 04): mỗi mô hình chạy trong một tiến trình riêng, kết quả cũ chỉ được dùng lại khi trùng **khoá chạy**, và dừng ô thì tắt hẳn tiến trình GPU. Chạy thử trước (2 truy vấn mỗi nhóm), rồi chạy toàn bộ: 624 truy vấn × 7 điều kiện × 2 mô hình.

In [8]:
def run_infer(model, out_dir, prompts_dir, smoke=False):
    log = LOGS / f"{model['tag']}{'_smoke' if smoke else ''}.log"
    cmd = [sys.executable, "ctidrift_infer.py", "--repo", model["repo"], "--revision", model["revision"],
           "--tag", model["tag"], "--prompts-dir", str(prompts_dir), "--conditions", ",".join(CONDITIONS),
           "--out-dir", str(out_dir), "--max-model-len", str(MAX_MODEL_LEN), "--max-tokens", str(MAX_TOKENS),
           "--gpu-mem", str(GPU_MEM), "--seed", str(SEED),
           "--chat-template-kwargs", json.dumps(model["chat_template_kwargs"])]
    print(f"▶ {model['tag']}: đang chạy… theo dõi tiến độ trong terminal bằng lệnh:\n    tail -f {log}")
    t0 = time.time()
    with open(log, "w", encoding="utf-8") as f:
        p = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT, env=ENV, cwd=ROOT, start_new_session=True)
        try:
            p.wait()
        except BaseException:
            def _kill(sig):
                try:
                    os.killpg(p.pid, sig)
                except ProcessLookupError:
                    pass
            _kill(signal.SIGTERM)
            try:
                p.wait(timeout=60)
            except BaseException:
                pass
            _kill(signal.SIGKILL)
            print("⏹ Đã dừng và giải phóng GPU. Kiểm tra lại bằng lệnh nvidia-smi nếu cần.")
            raise
    tail = log.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(l for l in tail if l.startswith(f"[{model['tag']}]") or "LỖI" in l or "Error" in l)[-3000:])
    print(f"◀ {model['tag']}: {'XONG' if p.returncode == 0 else 'LỖI'} sau {(time.time() - t0) / 60:.1f} phút")
    if p.returncode != 0:
        print("\n".join(tail[-40:]))
        raise RuntimeError(f"Suy luận lỗi — gửi thầy file {log}")

SMOKE = OUT / "raw_smoke"
for m in MODELS:
    run_infer(m, SMOKE, SMOKE_PROMPTS, smoke=True)
fmt_bad = []
for m in MODELS:
    for cond in CONDITIONS:
        for r in map(json.loads, open(SMOKE / m["tag"] / f"{cond}.jsonl", encoding="utf-8")):
            if score_drift(QBYID[r["query_id"]], r["text"], TACTIC_ALIAS)["format"] != "json":
                fmt_bad.append((m["tag"], cond, r["query_id"], r["text"][:80]))
assert not fmt_bad, f"Có câu trả lời không phải JSON — gửi thầy: {fmt_bad[:5]}"
print(f"✓ Chạy thử: mọi câu trả lời đúng định dạng JSON")

▶ qwen3-8b: đang chạy… theo dõi tiến độ trong terminal bằng lệnh:
    tail -f ~/p/cti-drift/results/ctx-18.1-19.2/logs/qwen3-8b_smoke.log
[qwen3-8b] mô hình tại /data/hf/hub/models--Qwen--Qwen3-8B-AWQ/snapshots/4da05a8edb55c6046cce958586c33b61da07bb79
[qwen3-8b] new: 14 prompt, dài nhất 1496 token (giới hạn 5984)
[qwen3-8b] old: 14 prompt, dài nhất 1488 token (giới hạn 5984)
[qwen3-8b] old_new: 14 prompt, dài nhất 1445 token (giới hạn 5984)
[qwen3-8b] new_old: 14 prompt, dài nhất 1445 token (giới hạn 5984)
[qwen3-8b] tag_old_new: 14 prompt, dài nhất 1518 token (giới hạn 5984)
[qwen3-8b] tag_new_old: 14 prompt, dài nhất 1518 token (giới hạn 5984)
[qwen3-8b] old_changelog: 14 prompt, dài nhất 1390 token (giới hạn 5984)
[qwen3-8b] new: xong 14 truy vấn trong 0.1 phút
[qwen3-8b] old: xong 14 truy vấn trong 0.0 phút
[qwen3-8b] old_new: xong 14 truy vấn trong 0.0 phút
[qwen3-8b] new_old: xong 14 truy vấn trong 0.0 phút
[qwen3-8b] tag_old_new: xong 14 truy vấn trong 0.0 phút
[qwen3-8b] tag_ne

In [9]:
for m in MODELS:
    run_infer(m, RAW, PROMPTS)

▶ qwen3-8b: đang chạy… theo dõi tiến độ trong terminal bằng lệnh:
    tail -f ~/p/cti-drift/results/ctx-18.1-19.2/logs/qwen3-8b.log
[qwen3-8b] mô hình tại /data/hf/hub/models--Qwen--Qwen3-8B-AWQ/snapshots/4da05a8edb55c6046cce958586c33b61da07bb79
[qwen3-8b] new: 624 prompt, dài nhất 1833 token (giới hạn 5984)
[qwen3-8b] old: 624 prompt, dài nhất 1834 token (giới hạn 5984)
[qwen3-8b] old_new: 624 prompt, dài nhất 1766 token (giới hạn 5984)
[qwen3-8b] new_old: 624 prompt, dài nhất 1766 token (giới hạn 5984)
[qwen3-8b] tag_old_new: 624 prompt, dài nhất 1839 token (giới hạn 5984)
[qwen3-8b] tag_new_old: 624 prompt, dài nhất 1839 token (giới hạn 5984)
[qwen3-8b] old_changelog: 624 prompt, dài nhất 1719 token (giới hạn 5984)
[qwen3-8b] new: xong 624 truy vấn trong 2.6 phút
[qwen3-8b] old: xong 624 truy vấn trong 2.7 phút
[qwen3-8b] old_new: xong 624 truy vấn trong 2.3 phút
[qwen3-8b] new_old: xong 624 truy vấn trong 2.3 phút
[qwen3-8b] tag_old_new: xong 624 truy vấn trong 2.4 phút
[qwen3-8b] 

## 6. Chấm điểm

Các cột giống notebook 04:

- `drift_ext`: trả lời theo bản cũ, gồm `STALE` hoặc trùng khớp đáp án bản cũ.
- `label_drift`: dùng tên tactic cũ.
- `subtype`: với câu D1 `changed`, cho biết đáp án mới có chứa tactic hoàn toàn mới (TA0112) hay chỉ chuyển giữa các tactic đã có.

In [10]:
OLD_TACTIC_IDS = {t for n, t in TACTIC_ALIAS.items() if n not in ALIAS_FILE["new_only_labels"]}
NEW_TACTIC_IDS = set(TACTIC_ALIAS.values()) - OLD_TACTIC_IDS

def subtype(q):
    if q["template"] != "D1_tech_tactic" or q["group"] != "changed":
        return ""
    return "NEW_TACTIC" if (set(q["gold"]) - set(q["other_gold"])) & NEW_TACTIC_IDS else "EXISTING_TACTICS"

rows = []
for m in MODELS:
    for cond in CONDITIONS:
        p = RAW / m["tag"] / f"{cond}.jsonl"
        got = [json.loads(l) for l in open(p, encoding="utf-8")]
        assert [r["query_id"] for r in got] == [q["query_id"] for q in QUERIES], f"{p}: thiếu hoặc sai thứ tự"
        for r in got:
            q = QBYID[r["query_id"]]
            s = score_drift(q, r["text"], TACTIC_ALIAS)
            pred, gold, other = set(s["pred"]), set(q["gold"]), set(q["other_gold"])
            other_exact = bool(pred) and pred == other and other != gold
            rows.append(dict(model=m["tag"], condition=cond, query_id=q["query_id"], item_id=q["item_id"],
                             lineage=q["lineage"], revoked=ITEMS[q["item_id"]]["anchor_status_new"] != "active",
                             template=q["template"], group=q["group"], pair_class=q["pair_class"],
                             framing=q["framing"], subtype=subtype(q), answer=r["text"],
                             finish_reason=r["finish_reason"], pred="|".join(s["pred"]), gold="|".join(q["gold"]),
                             other_gold="|".join(q["other_gold"]), em=s["em"], outcome=s["outcome"],
                             drift_ext=s["outcome"] in ("STALE", "ANACHRONISM") or other_exact,
                             label_drift=s["label_drift"], format=s["format"]))
SC = pd.DataFrame(rows)
SC.to_csv(OUT / "scored.csv", index=False)
chk = SC.groupby(["model", "condition"]).agg(n=("query_id", "count"), json=("format", lambda x: (x == "json").mean()),
                                             truncated=("finish_reason", lambda x: (x == "length").sum()))
assert (chk.n == len(QUERIES)).all()
chk

n  json  truncated
model       condition                          
llama3.1-8b new            624   1.0          0
            new_old        624   1.0          0
            old            624   1.0          0
            old_changelog  624   1.0          0
            old_new        624   1.0          0
            tag_new_old    624   1.0          0
            tag_old_new    624   1.0          0
qwen3-8b    new            624   1.0          0
            new_old        624   1.0          0
            old            624   1.0          0
            old_changelog  624   1.0          0
            old_new        624   1.0          0
            tag_new_old    624   1.0          0
            tag_old_new    624   1.0          0

## 7. Kết quả

Khoảng tin cậy 95% là **bootstrap theo cụm dòng dõi** (`lineage`, 2.000 lần lặp). So sánh giữa hai điều kiện là **ghép cặp**: cùng một tập truy vấn, cùng các chỉ số lấy mẫu lại.

**Mọi bảng dưới đây chỉ dùng cách hỏi `current`.** Riêng mục 7.6 báo cáo `unversioned`.

### 7.1. Bảng tổng theo điều kiện

In [11]:
CLUSTER = "lineage"           # cụm bootstrap: các câu về cùng kỹ thuật / mã cũ + mã thay thế

def _agg(df, col):
    g = df.groupby(CLUSTER)[col].agg(["sum", "count"])
    return g["sum"].to_numpy(float), g["count"].to_numpy(float), g.index

def boot_rate(df, col):
    if len(df) == 0:
        return (float("nan"),) * 3
    s, c, _ = _agg(df, col)
    idx = np.random.default_rng(SEED).integers(0, len(s), size=(BOOT_B, len(s)))
    r = s[idx].sum(1) / c[idx].sum(1)
    lo, hi = np.percentile(r, [2.5, 97.5])
    return (s.sum() / c.sum(), lo, hi)

def boot_diff_paired(df_a, df_b, col):
    sa, ca, ia = _agg(df_a, col)
    sb, cb, ib = _agg(df_b, col)
    assert list(ia) == list(ib), "hai nhóm phải có cùng tập mục"
    if len(sa) == 0:
        return (float("nan"),) * 3
    idx = np.random.default_rng(SEED).integers(0, len(sa), size=(BOOT_B, len(sa)))
    r = sa[idx].sum(1) / ca[idx].sum(1) - sb[idx].sum(1) / cb[idx].sum(1)
    lo, hi = np.percentile(r, [2.5, 97.5])
    return (sa.sum() / ca.sum() - sb.sum() / cb.sum(), lo, hi)

def boot_diff_indep(df_a, df_b, col):
    sa, ca, _ = _agg(df_a, col)
    sb, cb, _ = _agg(df_b, col)
    if len(sa) == 0 or len(sb) == 0:
        return (float("nan"),) * 3
    rng = np.random.default_rng(SEED)
    ia = rng.integers(0, len(sa), size=(BOOT_B, len(sa)))
    ib = rng.integers(0, len(sb), size=(BOOT_B, len(sb)))
    r = sa[ia].sum(1) / ca[ia].sum(1) - sb[ib].sum(1) / cb[ib].sum(1)
    lo, hi = np.percentile(r, [2.5, 97.5])
    return (sa.sum() / ca.sum() - sb.sum() / cb.sum(), lo, hi)

def fmt_b(t):
    v, lo, hi = t
    return "—" if v != v else f"{v:.2f} [{lo:.2f}, {hi:.2f}]"

FRAMING = "current"           # cách hỏi dùng để kiểm định (mục 7.6 báo cáo unversioned)

def sel(model, cond, group, template=None, framing=None, data=None):
    data = SC if data is None else data
    d = data[(data.model == model) & (data.condition == cond) & (data.group == group)
             & (data.framing == (framing or FRAMING))]
    if template:
        d = d[d.template == template]
    return d.sort_values("query_id")

T1 = []
for m in [x["tag"] for x in MODELS]:
    for c in CONDITIONS:
        T1.append({"model": m, "condition": c,
                   "changed: drift": fmt_b(boot_rate(sel(m, c, "changed"), "drift_ext")),
                   "changed: acc": fmt_b(boot_rate(sel(m, c, "changed"), "em")),
                   "label: tên cũ": fmt_b(boot_rate(sel(m, c, "label"), "label_drift")),
                   "label: acc": fmt_b(boot_rate(sel(m, c, "label"), "em")),
                   "stable: acc": fmt_b(boot_rate(sel(m, c, "stable"), "em"))})
T1 = pd.DataFrame(T1)
T1.to_csv(OUT / "summary_conditions.csv", index=False)
T1

,model,condition,changed: drift,changed: acc,label: tên cũ,label: acc,stable: acc
0,qwen3-8b,new,"0.02 [0.00, 0.04]","0.79 [0.70, 0.89]","0.00 [0.00, 0.00]","1.00 [1.00, 1.00]","1.00 [1.00, 1.00]"
1,qwen3-8b,old,"1.00 [1.00, 1.00]","0.00 [0.00, 0.00]","1.00 [1.00, 1.00]","1.00 [1.00, 1.00]","1.00 [1.00, 1.00]"
2,qwen3-8b,old_new,"0.94 [0.89, 0.98]","0.06 [0.02, 0.11]","0.97 [0.94, 1.00]","1.00 [1.00, 1.00]","1.00 [1.00, 1.00]"
3,qwen3-8b,new_old,"0.84 [0.76, 0.91]","0.16 [0.09, 0.24]","0.95 [0.90, 0.99]","0.99 [0.96, 1.00]","1.00 [1.00, 1.00]"
4,qwen3-8b,tag_old_new,"0.08 [0.02, 0.15]","0.91 [0.84, 0.98]","0.05 [0.01, 0.10]","1.00 [1.00, 1.00]","1.00 [1.00, 1.00]"
5,qwen3-8b,tag_new_old,"0.27 [0.15, 0.38]","0.71 [0.60, 0.83]","0.34 [0.24, 0.45]","0.99 [0.96, 1.00]","1.00 [1.00, 1.00]"
6,qwen3-8b,old_changelog,"0.13 [0.07, 0.19]","0.81 [0.74, 0.89]","0.66 [0.55, 0.76]","1.00 [1.00, 1.00]","1.00 [1.00, 1.00]"
7,llama3.1-8b,new,"0.10 [0.02, 0.21]","0.79 [0.68, 0.90]","0.00 [0.00, 0.00]","0.99 [0.96, 1.00]","0.85 [0.79, 0.91]"
8,llama3.1-8b,old,"1.00 [1.00, 1.00]","0.00 [0.00, 0.00]","1.00 [1.00, 1.00]","1.00 [1.00, 1.00]","0.85 [0.79, 0.91]"
9,llama3.1-8b,old_new,"0.93 [0.88, 0.97]","0.05 [0.02, 0.10]","0.96 [0.91, 1.00]","0.97 [0.94, 1.00]","0.92 [0.87, 0.97]"


### 7.2. Kiểm định các giả thuyết H1–H5

Mỗi dòng là một so sánh ghép cặp (H1–H4) hoặc giữa hai nhóm (H5). **Hỗ trợ** khi khoảng tin cậy nằm hẳn trên 0. Có 10 so sánh × 2 mô hình, nên khi viết bài phải nói rõ đây là nhiều so sánh định trước, và ưu tiên những kết quả **nhất quán ở cả hai mô hình**. H5 chỉ mang tính khám phá.

H3 được kiểm định riêng cho hai thứ tự (H3a: cũ trước, H3b: mới trước). H2 và H1 còn được tính riêng cho nhóm `label` (dùng tên cũ), vì đó là hiện tượng "cùng mã, đổi nghĩa".

In [12]:
HYP = [
    ("H1 vị trí",              "changed", "drift_ext",   "old_new",     "new_old"),
    ("H2 có mặt đoạn cũ",       "changed", "drift_ext",   "new_old",     "new"),
    ("H3a nhãn (cũ trước)",     "changed", "drift_ext",   "old_new",     "tag_old_new"),
    ("H3b nhãn (mới trước)",    "changed", "drift_ext",   "new_old",     "tag_new_old"),
    ("H4 ghi chú thay đổi",     "changed", "drift_ext",   "old",         "old_changelog"),
    ("H1-label vị trí",         "label",   "label_drift", "old_new",     "new_old"),
    ("H2-label có mặt đoạn cũ", "label",   "label_drift", "new_old",     "new"),
    ("H3-label nhãn",           "label",   "label_drift", "new_old",     "tag_new_old"),
]
H4_DATA = SC[~((SC.template == "D1_tech_tactic") & SC.revoked)]     # H4: loại D1 thu hồi (xem bảng giả thuyết)

def hypothesis_table(framing):
    out = []
    for m in [x["tag"] for x in MODELS]:
        for name, grp, col, a, b in HYP:
            data = H4_DATA if name.startswith("H4") else SC
            d = boot_diff_paired(sel(m, a, grp, framing=framing, data=data), sel(m, b, grp, framing=framing, data=data), col)
            out.append({"model": m, "giả thuyết": name, "so sánh": f"{col}({a}) − {col}({b})",
                        "hiệu": fmt_b(d), "hỗ trợ": "✓" if d[1] > 0 else ("ngược chiều" if d[2] < 0 else "·")})
        for c in ("new_old", "old_new"):
            d1 = SC[(SC.model == m) & (SC.condition == c) & (SC.subtype != "") & ~SC.revoked & (SC.framing == framing)]
            d = boot_diff_indep(d1[d1.subtype == "NEW_TACTIC"], d1[d1.subtype == "EXISTING_TACTICS"], "drift_ext")
            out.append({"model": m, "giả thuyết": f"H5 nhãn quen thuộc ({c}, khám phá)",
                        "so sánh": "drift(NEW_TACTIC) − drift(EXISTING_TACTICS), chỉ kỹ thuật còn hiệu lực",
                        "hiệu": fmt_b(d), "hỗ trợ": "✓" if d[1] > 0 else ("ngược chiều" if d[2] < 0 else "·")})
    return pd.DataFrame(out)

T2 = hypothesis_table("current")
T2.to_csv(OUT / "summary_hypotheses.csv", index=False)
T2

,model,giả thuyết,so sánh,hiệu,hỗ trợ
0,qwen3-8b,H1 vị trí,drift_ext(old_new) − drift_ext(new_old),"0.10 [0.03, 0.17]",✓
1,qwen3-8b,H2 có mặt đoạn cũ,drift_ext(new_old) − drift_ext(new),"0.82 [0.75, 0.89]",✓
2,qwen3-8b,H3a nhãn (cũ trước),drift_ext(old_new) − drift_ext(tag_old_new),"0.86 [0.78, 0.93]",✓
3,qwen3-8b,H3b nhãn (mới trước),drift_ext(new_old) − drift_ext(tag_new_old),"0.57 [0.46, 0.69]",✓
4,qwen3-8b,H4 ghi chú thay đổi,drift_ext(old) − drift_ext(old_changelog),"1.00 [1.00, 1.00]",✓
5,qwen3-8b,H1-label vị trí,label_drift(old_new) − label_drift(new_old),"0.03 [-0.04, 0.09]",·
6,qwen3-8b,H2-label có mặt đoạn cũ,label_drift(new_old) − label_drift(new),"0.95 [0.90, 0.99]",✓
7,qwen3-8b,H3-label nhãn,label_drift(new_old) − label_drift(tag_new_old),"0.61 [0.50, 0.71]",✓
8,qwen3-8b,"H5 nhãn quen thuộc (new_old, khám phá)","drift(NEW_TACTIC) − drift(EXISTING_TACTICS), c...","0.49 [0.32, 0.68]",✓
9,qwen3-8b,"H5 nhãn quen thuộc (old_new, khám phá)","drift(NEW_TACTIC) − drift(EXISTING_TACTICS), c...","0.23 [0.10, 0.39]",✓


### 7.3. Theo mẫu câu hỏi × mã bị thu hồi (nhóm `changed`)

D4 (mã còn hiệu lực không) chỉ trả lời đúng được khi ngữ cảnh **nối** mã cũ với mã mới. Đoạn của mã thay thế (N) không làm được việc đó; ghi chú thay đổi (L) thì có. Với D1 có mã bị thu hồi, ghi chú chỉ cho mã thay thế mà không cho tactic, nên `old_changelog` không thể đúng; vì vậy H4 loại nhóm này.

Bảng ghi `drift_ext / acc` cho từng ô: một điều kiện có thể giảm lỗi thời chỉ vì mô hình **bỏ trống** câu trả lời, nên phải xem cả độ chính xác.

In [13]:
CH = SC[(SC.group == "changed") & (SC.framing == FRAMING)]
_rows = [dict(template=t, revoked=r, condition=c, model=m,
              cell=f"{g.drift_ext.mean():.2f} / {g.em.mean():.2f} (n={len(g)})")
         for (t, r, c, m), g in CH.groupby(["template", "revoked", "condition", "model"])]
T3 = (pd.DataFrame(_rows).pivot(index=["template", "revoked", "condition"], columns="model", values="cell")
      .reindex(list(CONDITIONS), level="condition"))
T3.to_csv(OUT / "summary_by_template.csv")
T3

model                                           llama3.1-8b  \
template          revoked condition                           
D1_tech_tactic    False   new            0.00 / 0.97 (n=72)   
                          old            1.00 / 0.00 (n=72)   
                          old_new        0.93 / 0.07 (n=72)   
                          new_old        0.69 / 0.31 (n=72)   
                          tag_old_new    0.04 / 0.96 (n=72)   
                          tag_new_old    0.04 / 0.96 (n=72)   
                          old_changelog  0.07 / 0.93 (n=72)   
                  True    new            0.47 / 0.20 (n=15)   
                          old            1.00 / 0.00 (n=15)   
                          old_new        1.00 / 0.00 (n=15)   
                          new_old        1.00 / 0.00 (n=15)   
                          tag_old_new    0.13 / 0.87 (n=15)   
                          tag_new_old    1.00 / 0.00 (n=15)   
                          old_changelog  1.00 / 0.00 (n=15)   
D2_subtech_parent True    new             0.00 / 0.62 (n=8)   
                          old             1.00 / 0.00 (n=8)   
                          old_new         1.00 / 0.00 (n=8)   
                          new_old         0.88 / 0.00 (n=8)   
                          tag_old_new     0.12 / 0.25 (n=8)   
                          tag_new_old     0.38 / 0.00 (n=8)   
                          old_changelog   0.62 / 0.00 (n=8)   
D4_id_currency    True    new            0.24 / 0.65 (n=17)   
                          old            1.00 / 0.00 (n=17)   
                          old_new        0.82 / 0.06 (n=17)   
                          new_old        0.88 / 0.00 (n=17)   
                          tag_old_new    0.00 / 1.00 (n=17)   
                          tag_new_old    0.24 / 0.71 (n=17)   
                          old_changelog  0.00 / 1.00 (n=17)   

model                                              qwen3-8b  
template          revoked condition                          
D1_tech_tactic    False   new            0.00 / 0.86 (n=72)  
                          old            1.00 / 0.00 (n=72)  
                          old_new        0.90 / 0.10 (n=72)  
                          new_old        0.76 / 0.24 (n=72)  
                          tag_old_new    0.00 / 1.00 (n=72)  
                          tag_new_old    0.10 / 0.88 (n=72)  
                          old_changelog  0.00 / 0.97 (n=72)  
                  True    new            0.00 / 0.60 (n=15)  
                          old            1.00 / 0.00 (n=15)  
                          old_new        1.00 / 0.00 (n=15)  
                          new_old        1.00 / 0.00 (n=15)  
                          tag_old_new    0.27 / 0.73 (n=15)  
                          tag_new_old    0.93 / 0.07 (n=15)  
                          old_changelog  1.00 / 0.00 (n=15)  
D2_subtech_parent True    new             0.00 / 0.88 (n=8)  
                          old             1.00 / 0.00 (n=8)  
                          old_new         1.00 / 0.00 (n=8)  
                          new_old         0.88 / 0.12 (n=8)  
                          tag_old_new     0.12 / 0.88 (n=8)  
                          tag_new_old     0.25 / 0.75 (n=8)  
                          old_changelog   0.00 / 0.50 (n=8)  
D4_id_currency    True    new            0.12 / 0.65 (n=17)  
                          old            1.00 / 0.00 (n=17)  
                          old_new        1.00 / 0.00 (n=17)  
                          new_old        1.00 / 0.00 (n=17)  
                          tag_old_new    0.24 / 0.71 (n=17)  
                          tag_new_old    0.41 / 0.59 (n=17)  
                          old_changelog  0.00 / 1.00 (n=17)

### 7.4. Đối chứng: gắn nhãn và trùng lặp đoạn có làm hại câu không đổi không?

Với nhóm `stable`, O và N gần như trùng nội dung. Độ chính xác nên **giữ nguyên** giữa các điều kiện. Nếu gắn nhãn phiên bản làm giảm độ chính xác ở đây, phương pháp có tác dụng phụ.

In [14]:
T4 = []
for m in [x["tag"] for x in MODELS]:
    for c in ("old_new", "tag_old_new", "new_old", "tag_new_old"):
        T4.append({"model": m, "condition": c, "stable acc": fmt_b(boot_rate(sel(m, c, "stable"), "em")),
                   "Δ so với new": fmt_b(boot_diff_paired(sel(m, c, "stable"), sel(m, "new", "stable"), "em"))})
pd.DataFrame(T4)

,model,condition,stable acc,Δ so với new
0,qwen3-8b,old_new,"1.00 [1.00, 1.00]","0.00 [0.00, 0.00]"
1,qwen3-8b,tag_old_new,"1.00 [1.00, 1.00]","0.00 [0.00, 0.00]"
2,qwen3-8b,new_old,"1.00 [1.00, 1.00]","0.00 [0.00, 0.00]"
3,qwen3-8b,tag_new_old,"1.00 [1.00, 1.00]","0.00 [0.00, 0.00]"
4,llama3.1-8b,old_new,"0.92 [0.87, 0.97]","0.07 [0.01, 0.14]"
5,llama3.1-8b,tag_old_new,"0.93 [0.89, 0.97]","0.08 [0.02, 0.15]"
6,llama3.1-8b,new_old,"0.93 [0.87, 0.97]","0.08 [0.02, 0.14]"
7,llama3.1-8b,tag_new_old,"0.91 [0.86, 0.96]","0.06 [0.00, 0.13]"


### 7.5. Phân bố kết quả chi tiết (nhóm `changed`, cách hỏi `current`)

In [15]:
T5 = pd.crosstab([CH.model, CH.condition], CH.outcome, normalize="index").round(3)
T5.to_csv(OUT / "summary_outcomes.csv")
T5

outcome                    CORRECT  NO_ANSWER  PARTIAL  STALE  WRONG
model       condition                                               
llama3.1-8b new              0.795      0.000    0.107  0.098  0.000
            new_old          0.196      0.000    0.027  0.777  0.000
            old              0.000      0.000    0.045  0.955  0.000
            old_changelog    0.750      0.000    0.000  0.223  0.027
            old_new          0.054      0.000    0.027  0.920  0.000
            tag_new_old      0.723      0.000    0.054  0.223  0.000
            tag_old_new      0.902      0.000    0.036  0.054  0.009
qwen3-8b    new              0.795      0.089    0.098  0.018  0.000
            new_old          0.161      0.000    0.009  0.830  0.000
            old              0.000      0.000    0.045  0.955  0.000
            old_changelog    0.812      0.000    0.018  0.134  0.036
            old_new          0.062      0.000    0.009  0.929  0.000
            tag_new_old      0.714      0.000    0.018  0.268  0.000
            tag_old_new      0.911      0.000    0.009  0.080  0.000

### 7.6. Cách hỏi `unversioned` (báo cáo riêng)

Câu hỏi không nói phiên bản. Khi ngữ cảnh chứa cả hai phiên bản, trả lời theo bản cũ không hẳn là sai. Bảng này để so sánh với 7.2, không dùng để kết luận giả thuyết.

In [16]:
T6 = hypothesis_table("unversioned")
T6.to_csv(OUT / "summary_hypotheses_unversioned.csv", index=False)
T6

,model,giả thuyết,so sánh,hiệu,hỗ trợ
0,qwen3-8b,H1 vị trí,drift_ext(old_new) − drift_ext(new_old),"0.09 [0.01, 0.18]",✓
1,qwen3-8b,H2 có mặt đoạn cũ,drift_ext(new_old) − drift_ext(new),"0.78 [0.70, 0.85]",✓
2,qwen3-8b,H3a nhãn (cũ trước),drift_ext(old_new) − drift_ext(tag_old_new),"0.13 [0.08, 0.19]",✓
3,qwen3-8b,H3b nhãn (mới trước),drift_ext(new_old) − drift_ext(tag_new_old),"0.36 [0.28, 0.44]",✓
4,qwen3-8b,H4 ghi chú thay đổi,drift_ext(old) − drift_ext(old_changelog),"0.66 [0.57, 0.74]",✓
5,qwen3-8b,H1-label vị trí,label_drift(old_new) − label_drift(new_old),"0.18 [0.10, 0.26]",✓
6,qwen3-8b,H2-label có mặt đoạn cũ,label_drift(new_old) − label_drift(new),"0.82 [0.74, 0.90]",✓
7,qwen3-8b,H3-label nhãn,label_drift(new_old) − label_drift(tag_new_old),"0.29 [0.19, 0.39]",✓
8,qwen3-8b,"H5 nhãn quen thuộc (new_old, khám phá)","drift(NEW_TACTIC) − drift(EXISTING_TACTICS), c...","0.33 [0.14, 0.53]",✓
9,qwen3-8b,"H5 nhãn quen thuộc (old_new, khám phá)","drift(NEW_TACTIC) − drift(EXISTING_TACTICS), c...","0.16 [0.03, 0.29]",✓


## 8. Ghi manifest và đóng gói gửi thầy

In [17]:
MANIFEST["n_queries"] = len(QUERIES)
MANIFEST["runs"] = {}
for m in MODELS:
    for cond in CONDITIONS:
        mp = RAW / m["tag"] / f"meta_{cond}.json"
        if mp.exists():
            MANIFEST["runs"][f"{m['tag']}/{cond}"] = json.loads(mp.read_text())
MANIFEST["outputs"] = {str(p.relative_to(OUT)): sha256(p) for p in sorted(OUT.rglob("*"))
                       if p.is_file() and p.suffix in (".csv", ".jsonl") and "raw_smoke" not in p.parts
                       and not p.parts[len(OUT.parts)].startswith("prompts")}
(OUT / "manifest.json").write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False))
ZIP = OUT / "gui_thay.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file() and p != ZIP and not p.parts[len(OUT.parts)].startswith("prompts_smoke"):
            z.write(p, p.relative_to(OUT))
print(f"Đã tạo {ZIP.relative_to(ROOT)} ({ZIP.stat().st_size / 1e6:.1f} MB)")

Đã tạo results/ctx-18.1-19.2/gui_thay.zip (2.7 MB)


## 9. Gửi thầy

1. Notebook đã chạy: `05_ngu_canh_kiem_soat.ipynb`.
2. File `results/ctx-18.1-19.2/gui_thay.zip`. Lần này file nén **có cả prompt**, để thầy kiểm tra lại từng ngữ cảnh.

Kết quả notebook này quyết định **phương pháp của Trụ 3** dựa trên cơ chế nào: thứ tự, nhãn phiên bản, hay nhật ký thay đổi. Sau đó ta kiểm định lại trên các lần chuyển phiên bản khác.